# 02 · Logística, KNN y SVM sobre Wine Quality: métricas, umbral y desbalance

**Módulo 4 · Sesión 9** — Clasificación

## Objetivos

El notebook 01 construyó la regresión logística desde cero y cerró con una advertencia: la
accuracy no sirve para evaluar un clasificador cuando las clases están desbalanceadas. Este
notebook construye el marco de evaluación completo sobre un dataset real, y de paso compara
tres familias de clasificadores:

1. Medir por qué la **accuracy engaña** con un 19 % de positivos, y reemplazarla por la
   matriz de confusión, precisión, recall y F1.
2. Detectar y **medir** una fuga que viene de fábrica en el dataset —filas duplicadas—
   antes de que haga ganar al modelo equivocado.
3. Leer las curvas **ROC** y **precisión-recall**, y entender por qué la segunda es más
   informativa aquí.
4. Ajustar el **umbral** de decisión con probabilidades de validación cruzada, según el
   costo de cada tipo de error.
5. Comparar tres estrategias frente al **desbalance** —pesos de clase, SMOTE y mover el
   umbral— y ver qué gana de verdad cada una.
6. Extender a **multiclase** (la calidad original, de 3 a 9) con softmax y macro-F1.

La teoría está en `02-knn-y-svm.md` y `03-metricas-clasificacion.md`.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `imbalanced-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as PipelineImb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEMILLA = 42

## 1. Los datos: Wine Quality

6497 vinos portugueses (1599 tintos, 4898 blancos) con 11 medidas fisicoquímicas y una
calificación sensorial `quality` de 3 a 9, mediana de al menos tres catadores (Cortez et
al., 2009). El problema **binario** del módulo: predecir si un vino es "bueno"
(`quality` $\geq 7$) a partir de la química. Hay una sola variable categórica (`tipo`), que
se codifica como 0/1.

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv")
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)  # 1 = tinto, 0 = blanco

print(vinos["quality"].value_counts().sort_index().rename("vinos por calidad").to_frame().T)
vinos["buena"] = (vinos["quality"] >= 7).astype(int)
print(f"\nProporción de vinos buenos: {vinos['buena'].mean():.3f}")
vinos.describe().round(2).T

Un 19.7 % de positivos: desbalance moderado, suficiente para que la accuracy engañe.

## 2. Antes de partir: 1177 filas idénticas

`01-diagnostico-eda-aplicado.ipynb` (módulo 2) encontró 107 filas idénticas en el Titanic
y concluyó que **no** eran duplicados: sin columna de identificador, dos pasajeros de
tercera clase con la misma edad y tarifa son perfectamente posibles. Aquí la situación es
distinta: son 11 mediciones de laboratorio con varios decimales cada una. Dos vinos
distintos con exactamente la misma química son mucho menos plausibles que **el mismo vino
registrado varias veces** (el dataset no tiene identificador de lote).

Más importante que el argumento es la **medición**. Si el mismo vino cae en entrenamiento y
en prueba, cualquier modelo que memorice tiene un acierto gratis. Lo comprobamos con el
modelo que mejor memoriza, KNN con $k=1$, sobre los datos con y sin duplicados.

In [ ]:
X_con_dup = vinos.drop(columns=["quality", "buena"])
y_con_dup = vinos["buena"]

print(f"Filas idénticas (contando todas las columnas): {vinos.duplicated().sum()}")
print(f"Filas con la misma química pero distinta calidad: "
      f"{X_con_dup.duplicated().sum() - vinos.duplicated().sum()}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)


def evaluar_cv(modelo, X, y, cv=cv):
    metricas = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
    r = cross_validate(modelo, X, y, cv=cv, scoring=metricas)
    return pd.Series({m: r[f"test_{m}"].mean() for m in metricas})


knn1 = Pipeline([("escalar", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=1))])

sin_dup = vinos.drop_duplicates().reset_index(drop=True)
X_sin_dup = sin_dup.drop(columns=["quality", "buena"])
y_sin_dup = sin_dup["buena"]

comparacion_dup = pd.DataFrame(
    {
        "con duplicados": evaluar_cv(knn1, X_con_dup, y_con_dup),
        "sin duplicados": evaluar_cv(knn1, X_sin_dup, y_sin_dup),
    }
)
print(f"\nKNN (k=1), validación cruzada estratificada de 5 pliegues:")
print(comparacion_dup.round(3))

Con los duplicados dentro, KNN con $k=1$ tiene un F1 de 0.66 — como se verá en la sección
4, **muy por encima de la regresión logística y de la SVM**. Sin ellos, cae a 0.47. La diferencia
no es que el modelo aprenda algo distinto: es que, con duplicados, el vecino más cercano
de muchas filas de validación es *la misma fila* en entrenamiento, a distancia cero. Es la
fuga por "gemelos" entre entrenamiento y prueba, y favorece sistemáticamente a los modelos
que memorizan sobre los que generalizan.

Decisión: se trabaja **sin duplicados** de aquí en adelante (5320 vinos, 19.0 % positivos),
y la partición estratificada se hace una sola vez, sobre esos datos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sin_dup, y_sin_dup, test_size=0.2, stratify=y_sin_dup, random_state=SEMILLA
)
print(f"Entrenamiento: {X_train.shape[0]} vinos ({y_train.mean():.3f} positivos)")
print(f"Prueba:        {X_test.shape[0]} vinos ({y_test.mean():.3f} positivos)")

## 3. Tres familias de clasificadores y la referencia trivial

- **Regresión logística**: frontera lineal en el espacio de las variables (notebook 01).
- **KNN**: vota entre los $k$ vecinos más cercanos; frontera local, sin parámetros que
  ajustar pero sensible a la escala y a $k$.
- **SVM con kernel RBF**: maximiza el margen en un espacio transformado; frontera no lineal.

Los tres necesitan las variables **estandarizadas** —KNN y SVM porque usan distancias;
la logística porque la regularización $L_2$ por defecto penaliza más a las variables de
escala pequeña (`04-regularizacion.md`)—, así que van dentro de un `Pipeline`.

Y la referencia trivial de siempre: predecir "no es buena" para todos los vinos.

In [ ]:
def crear_pipeline(clasificador):
    return Pipeline([("escalar", StandardScaler()), ("clf", clasificador)])


modelos = {
    "Logística": crear_pipeline(LogisticRegression(max_iter=2000)),
    "KNN (k=15)": crear_pipeline(KNeighborsClassifier(n_neighbors=15)),
    "SVM RBF (C=10)": crear_pipeline(SVC(C=10, probability=True, random_state=SEMILLA)),
}

referencia_accuracy = 1 - y_train.mean()
print(f"Referencia trivial ('ninguno es bueno'): accuracy = {referencia_accuracy:.3f}")

resultados = pd.DataFrame({nombre: evaluar_cv(m, X_train, y_train) for nombre, m in modelos.items()}).T
print(resultados[["accuracy"]].round(3))

> `probability=True` en la SVM ajusta internamente una calibración de Platt (una regresión
> logística sobre la distancia al margen) para producir probabilidades; sin ella, `SVC` solo
> da la distancia con signo (`decision_function`). Es 🔵 opcional entender cómo, pero
> necesario para comparar umbrales en la misma escala que los otros dos modelos.

Los tres modelos tienen una accuracy entre 0.82 y 0.84. La referencia trivial, **0.81**.
Con este número no se puede distinguir un modelo bueno de uno que no aprendió nada — que es
exactamente lo que pasa con una SVM de kernel lineal, que predice "no" para todos los vinos
y obtiene 0.80 sin producir un solo positivo:

In [ ]:
svm_lineal = crear_pipeline(SVC(kernel="linear", random_state=SEMILLA))
pred_lineal = cross_val_predict(svm_lineal, X_train, y_train, cv=cv)
print(f"SVM lineal — accuracy: {np.mean(pred_lineal == y_train):.3f}   positivos predichos: {pred_lineal.sum()}")

## 4. La matriz de confusión y las métricas que salen de ella

Con $\hat{y}$ binaria, todo se resume en cuatro números: verdaderos positivos (VP), falsos
positivos (FP), falsos negativos (FN) y verdaderos negativos (VN). De ahí:

$$
\text{precisión} = \frac{VP}{VP + FP} \qquad
\text{recall} = \frac{VP}{VP + FN} \qquad
F_1 = \frac{2 \cdot \text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}
$$

Para obtener predicciones honestas sobre **todo** el conjunto de entrenamiento —cada fila
predicha por un modelo que no la vio— se usa `cross_val_predict`: es lo que permite
construir una sola matriz de confusión a partir de la validación cruzada.

In [ ]:
proba_cv = {}
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.8))
for eje, (nombre, modelo) in zip(ejes, modelos.items()):
    proba_cv[nombre] = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    pred = (proba_cv[nombre] >= 0.5).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_train, pred), display_labels=["no buena", "buena"]).plot(ax=eje, colorbar=False)
    eje.set_title(nombre)
plt.tight_layout()
plt.show()

print(resultados[["accuracy", "precision", "recall", "f1"]].round(3))

Ahora sí hay diferencias. Los tres modelos tienen una precisión cercana a 0.6 (de cada 10
vinos que declaran buenos, 6 lo son), pero un recall de solo 0.31–0.34: **se les escapan
dos de cada tres vinos buenos**. La accuracy de 0.83 escondía que el modelo acierta casi
exclusivamente en la clase mayoritaria.

Fíjese también en que KNN con $k=1$ tenía un F1 de 0.66 *con duplicados* (sección 2) —
habría "ganado" esta tabla por un margen amplio sobre el 0.40–0.44 de los tres modelos. La
fuga no solo inflaba un número: cambiaba qué modelo se elige.

## 5. Curvas ROC y precisión-recall

Las métricas de la sección 4 dependen del umbral 0.5, que nadie eligió. Las curvas ROC y
PR muestran el comportamiento **para todos los umbrales** a la vez:

- **ROC**: tasa de verdaderos positivos (recall) contra tasa de falsos positivos
  ($FP / (FP + VN)$). El área bajo la curva (AUC-ROC) es la probabilidad de que un positivo
  al azar reciba mayor puntuación que un negativo al azar. Un clasificador aleatorio da 0.5.
- **PR**: precisión contra recall. Su área (*average precision*, AP) tiene como referencia
  aleatoria la **prevalencia**, aquí 0.19.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.6))
for nombre, p in proba_cv.items():
    fpr, tpr, _ = roc_curve(y_train, p)
    prec, rec, _ = precision_recall_curve(y_train, p)
    ejes[0].plot(fpr, tpr, label=f"{nombre} (AUC = {roc_auc_score(y_train, p):.3f})")
    ejes[1].plot(rec, prec, label=f"{nombre} (AP = {average_precision_score(y_train, p):.3f})")

ejes[0].plot([0, 1], [0, 1], "k--", lw=0.8, label="aleatorio (0.5)")
ejes[0].set_xlabel("Tasa de falsos positivos")
ejes[0].set_ylabel("Recall (tasa de verdaderos positivos)")
ejes[0].set_title("Curva ROC")
ejes[0].legend(fontsize=8)

ejes[1].axhline(y_train.mean(), color="k", ls="--", lw=0.8, label=f"aleatorio (prevalencia = {y_train.mean():.2f})")
ejes[1].set_xlabel("Recall")
ejes[1].set_ylabel("Precisión")
ejes[1].set_title("Curva precisión-recall")
ejes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Las dos curvas cuentan historias distintas del mismo modelo. El AUC-ROC de 0.80–0.83 parece
"bueno"; la curva PR muestra que, para recuperar el 80 % de los vinos buenos, la precisión
cae a 0.34–0.41. La razón es aritmética: la tasa de falsos positivos del eje $x$ de la ROC
se divide entre los **3449 negativos**, así que 900 falsos positivos la mueven solo hasta
0.26; esos mismos 900 falsos positivos, frente a los ≈650 verdaderos positivos que hacen
falta para un recall de 0.8 (hay 807 vinos buenos), hunden la precisión a 0.4. **Con clases
desbalanceadas, la curva PR es la que muestra el problema**, y la AP es la métrica de
resumen que conviene reportar junto al AUC-ROC.

## 6. Elegir el umbral

`predict` usa 0.5 porque sí. La decisión correcta depende de **qué error cuesta más**, y se
toma sobre las probabilidades de validación cruzada — no sobre el conjunto de prueba, que
se reserva para el final. Primero, cómo se mueven precisión, recall y F1 al recorrer el
umbral para la regresión logística:

In [ ]:
p_log = proba_cv["Logística"]
umbrales = np.round(np.linspace(0.05, 0.95, 91), 2)
curvas = pd.DataFrame(
    {
        "umbral": umbrales,
        "precisión": [precision_score(y_train, p_log >= u, zero_division=0) for u in umbrales],
        "recall": [recall_score(y_train, p_log >= u) for u in umbrales],
        "F1": [f1_score(y_train, p_log >= u) for u in umbrales],
    }
)
mejor = curvas.loc[curvas["F1"].idxmax()]

plt.figure(figsize=(7, 4.3))
for col in ["precisión", "recall", "F1"]:
    plt.plot(curvas["umbral"], curvas[col], label=col)
plt.axvline(0.5, color="gray", ls="--", lw=0.8, label="umbral por defecto")
plt.axvline(mejor["umbral"], color="C2", ls=":", lw=1.2, label=f"máximo F1 (u = {mejor['umbral']:.2f})")
plt.xlabel("Umbral sobre P(buena)")
plt.title("Regresión logística: métricas según el umbral (probabilidades de CV)")
plt.legend()
plt.show()
print(mejor.round(3))

Bajar el umbral de 0.5 a ≈0.27 sube el F1 de 0.41 a 0.55: el recall pasa de 0.31 a 0.67 y
la precisión baja de 0.59 a 0.47. No hay un umbral "correcto": maximizar F1 es solo una
forma de pesar los dos errores por igual.

### Con costos explícitos

Supongamos que el modelo se usa para decidir qué vinos enviar a una cata de confirmación
(cara). Un falso positivo cuesta una cata desperdiciada; un falso negativo cuesta un vino
bueno que se vende como corriente — digamos, **tres** veces más caro. El umbral óptimo es
el que minimiza el costo total esperado:

In [ ]:
COSTO_FP, COSTO_FN = 1, 3


def costo_total(y, p, umbral):
    pred = p >= umbral
    fp = np.sum(pred & (y == 0))
    fn = np.sum(~pred & (y == 1))
    return COSTO_FP * fp + COSTO_FN * fn


costos = pd.DataFrame(
    {nombre: [costo_total(y_train.to_numpy(), p, u) for u in umbrales] for nombre, p in proba_cv.items()},
    index=umbrales,
)

plt.figure(figsize=(7, 4.3))
for nombre in costos.columns:
    plt.plot(costos.index, costos[nombre], label=nombre)
plt.axvline(0.5, color="gray", ls="--", lw=0.8)
plt.xlabel("Umbral")
plt.ylabel(f"Costo total (FP × {COSTO_FP} + FN × {COSTO_FN})")
plt.title("Costo esperado según el umbral, sobre las probabilidades de CV")
plt.legend()
plt.show()

umbral_optimo = costos.idxmin()
print("Umbral de mínimo costo por modelo:")
print(pd.DataFrame({"umbral": umbral_optimo, "costo": costos.min(), "costo en 0.5": costos.loc[0.5]}).round(3))

Con esa estructura de costos, el umbral óptimo de los tres modelos está entre 0.19 y 0.27,
muy por debajo de 0.5, y usar el valor por defecto sale entre un 21 y un 32 % más caro. El
umbral no es un detalle de implementación: **es donde la métrica de negocio entra en el
modelo**.

## 7. Tres estrategias frente al desbalance, medidas

La literatura ofrece tres remedios para las clases desbalanceadas, y conviene saber qué
hace cada uno antes de aplicarlos por reflejo:

1. **Mover el umbral** (sección 6): no toca el modelo, solo la decisión.
2. **Pesos de clase** (`class_weight="balanced"`): cada error en la clase minoritaria pesa
   $n / (2 n_{\text{minoritaria}})$ veces más en la función de costo.
3. **Sobremuestreo sintético** (SMOTE): crea positivos nuevos interpolando entre vecinos, hasta
   igualar las clases. Debe ir **dentro** del pipeline —solo sobre los pliegues de
   entrenamiento— o los positivos sintéticos contaminan la validación (`imblearn.pipeline`
   lo garantiza; el `Pipeline` de `scikit-learn` no acepta remuestreadores).

Comparamos las tres sobre la logística y la SVM, con dos métricas independientes del umbral
(AUC-ROC y AP) y con el mejor F1 alcanzable moviendo el umbral.

In [ ]:
def mejor_f1(y, p):
    prec, rec, _ = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return f1.max()


variantes = {
    "Logística": crear_pipeline(LogisticRegression(max_iter=2000)),
    "Logística + pesos": crear_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Logística + SMOTE": PipelineImb([
        ("escalar", StandardScaler()),
        ("smote", SMOTE(random_state=SEMILLA)),
        ("clf", LogisticRegression(max_iter=2000)),
    ]),
    "SVM": crear_pipeline(SVC(C=10, probability=True, random_state=SEMILLA)),
    "SVM + pesos": crear_pipeline(SVC(C=10, probability=True, class_weight="balanced", random_state=SEMILLA)),
    "SVM + SMOTE": PipelineImb([
        ("escalar", StandardScaler()),
        ("smote", SMOTE(random_state=SEMILLA)),
        ("clf", SVC(C=10, probability=True, random_state=SEMILLA)),
    ]),
}

filas = []
for nombre, modelo in variantes.items():
    p = cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    filas.append(
        {
            "modelo": nombre,
            "AUC-ROC": roc_auc_score(y_train, p),
            "AP": average_precision_score(y_train, p),
            "F1 en 0.5": f1_score(y_train, p >= 0.5),
            "recall en 0.5": recall_score(y_train, p >= 0.5),
            "mejor F1 (umbral libre)": mejor_f1(y_train, p),
        }
    )
tabla_desbalance = pd.DataFrame(filas).set_index("modelo")
print(tabla_desbalance.round(3).to_string())

La tabla separa dos cosas que suelen confundirse:

- En la **logística**, pesos y SMOTE disparan el recall en 0.5 (de 0.31 a ≈0.78) y con
  ello el F1 en 0.5. Pero AUC-ROC (0.83), AP (0.52) y el *mejor* F1 alcanzable (0.55) **no
  cambian** en la tercera cifra. Ambas técnicas equivalen a mover el umbral: reordenan poco
  o nada a los vinos, solo desplazan la frontera. Si el umbral se va a elegir de todos
  modos (sección 6), no aportan.
- En la **SVM** los pesos sí cambian el modelo — y las métricas discrepan sobre si para
  mejor: el AUC-ROC sube de 0.80 a 0.83 pero la AP **baja** de 0.53 a 0.51, y el mejor F1
  se mueve de 0.540 a 0.547. La SVM no estima probabilidades: su frontera la fijan los
  vectores de soporte, y darle más peso a la clase minoritaria cambia *cuáles* puntos lo
  son. Es un modelo distinto, no una decisión distinta; que sea mejor depende de qué
  métrica se mire, igual que RMSE y MAE discreparon en el módulo 3.

La regla práctica que sale de medir: **antes de remuestrear, mover el umbral**. Si con eso
no basta, pesos de clase (baratos, sin datos sintéticos). SMOTE es el último recurso, y su
efecto debe comprobarse con AP o AUC, nunca con el F1 en 0.5 — que es justo la métrica que
siempre "mejora" y por eso se cita.

## 8. Multiclase: la calidad original con softmax

La sección 6 del notebook 01 implementó softmax a mano; `LogisticRegression` lo usa
automáticamente cuando el objetivo tiene más de dos clases. Aquí el objetivo es `quality`
tal cual (3 a 9), con la misma partición (`X_train` ya no lleva `quality`, así que basta con
recuperar la columna por índice).

In [ ]:
y_train_multi = sin_dup.loc[X_train.index, "quality"]

softmax = crear_pipeline(LogisticRegression(max_iter=5000))
pred_multi = cross_val_predict(softmax, X_train, y_train_multi, cv=cv)

print(classification_report(y_train_multi, pred_multi, zero_division=0))

La accuracy global es 0.55, y las clases 8 y 9 tienen recall **cero**: el modelo nunca las
predice (y la 3 y la 4, casi nunca). Con 7 clases tan desbalanceadas (5 vinos de calidad 9
frente a 1832 de calidad 6), la métrica que refleja eso es el **macro-F1** (promedio simple
del F1 de cada clase, 0.25), no la accuracy ni el F1 ponderado (0.52), que vuelven a premiar
acertar la clase grande. Pero la matriz de confusión muestra que la mayoría de los errores
no son arbitrarios:

In [ ]:
matriz_multi = pd.crosstab(y_train_multi, pred_multi, rownames=["real"], colnames=["predicha"])
print(matriz_multi)

distancia = np.abs(y_train_multi.to_numpy() - pred_multi)
print(f"\nErrores a exactamente un punto de calidad: {np.mean(distancia == 1):.3f}")
print(f"Errores a dos o más puntos:                {np.mean(distancia >= 2):.3f}")
print(f"Aciertos exactos:                          {np.mean(distancia == 0):.3f}")

De los errores, el 88 % (0.40 de 0.45) están a **un punto** de la calidad real. La calidad es una variable
**ordinal** —7 está más cerca de 6 que de 3— y softmax la trata como categorías sin orden.
Eso sugiere que el problema binario "$\geq 7$" (o incluso una regresión sobre `quality`,
con las herramientas del módulo 3) captura mejor la estructura que un softmax de 7 clases.
Elegir cómo formular el problema es una decisión anterior a elegir el modelo.

## 9. Sobre el conjunto de prueba

Todas las decisiones anteriores —quitar duplicados, elegir modelo, elegir umbral— se tomaron
con validación cruzada sobre `X_train`. Ahora, una sola vez, el conjunto de prueba, con el
umbral de mínimo costo de la sección 6 para la logística.

In [ ]:
final = crear_pipeline(LogisticRegression(max_iter=2000)).fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]
u = umbral_optimo["Logística"]
pred_test = (p_test >= u).astype(int)

print(f"Umbral usado: {u:.2f}")
print(f"AUC-ROC: {roc_auc_score(y_test, p_test):.3f}   AP: {average_precision_score(y_test, p_test):.3f}")
print(f"Precisión: {precision_score(y_test, pred_test):.3f}   Recall: {recall_score(y_test, pred_test):.3f}   "
      f"F1: {f1_score(y_test, pred_test):.3f}")
print(f"Costo en prueba con umbral {u:.2f}: {costo_total(y_test.to_numpy(), p_test, u)}   "
      f"con 0.5: {costo_total(y_test.to_numpy(), p_test, 0.5)}")
print(confusion_matrix(y_test, pred_test))

## Resumen

| Lo que se midió | Resultado |
|---|---|
| Accuracy de los tres modelos vs. referencia trivial | 0.82–0.84 vs. 0.81: no distingue nada |
| Duplicados entre entrenamiento y validación | F1 de KNN ($k=1$) de 0.66 → 0.47 al quitarlos: la fuga elegía al modelo equivocado |
| ROC vs. PR | AUC-ROC ≈ 0.8 "parece bien"; la PR muestra que un recall de 0.8 cuesta una precisión de 0.34–0.41 |
| Umbral | El óptimo por costo está en 0.19–0.27; usar 0.5 sale 21–32 % más caro |
| Pesos de clase / SMOTE en la logística | Suben el recall en 0.5, pero no AUC, ni AP, ni el mejor F1: equivalen a mover el umbral |
| Pesos de clase en la SVM | Cambian el modelo; AUC sube y AP baja: las métricas discrepan |
| Softmax sobre 7 clases | Macro-F1 0.25; el 88 % de los errores a ±1 punto: el problema es ordinal |
| Conjunto de prueba (logística, umbral 0.27) | AUC 0.82, AP 0.48, F1 0.52; costo 383 frente a 488 con el umbral 0.5 |

Ningún modelo de esta sesión supera un AP de ≈0.53 en validación cruzada. La sesión 10
introduce los árboles y los ensambles sobre este mismo dataset y esta misma partición, y
la sesión 11 explica qué variables usan para conseguir lo que consiguen.